In [3]:
import pandas as pd
import os 
import sys
sys.path.append('../src')

import numpy as np

from preprocessing import get_dfs, create_static_df, create_notes_df
from tqdm import tqdm
from models import NotesEncoder
import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = NotesEncoder(model_name="../models/med_gte_simcse_en_ger").to(device)
encoder.model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 1024, padding_idx=0)
    (position_embeddings): Embedding(512, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-23): 24 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

In [ ]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')
#notes = create_notes_df(dfs, filename=None)
notes

Reading notes from exams.csv
Found 206509 texts
Loading 215137 texts from clinical assessments
Concatenated texts and deleted NaNs, final count: 367140
Average texts per patient: 106.9
Reading notes from exams.csv
Found 206509 texts
Loading 215137 texts from clinical assessments
Concatenated texts and deleted NaNs, final count: 367140
Average texts per patient: 106.9
367140
367140


,patient_id,transplant_id,rel_days,text,type,embeddings
0,33,1805,-1,Thorax bed side vom 18.12.2003 19:45 : Herz im...,Röntgen,"[-0.00928636, -0.021458145, 0.029821012, -0.00..."
1,33,1805,0,Farbkodierte Dopplersonographie und Power-Dop...,Sono,"[0.0007001866, 0.026864352, -0.00019196428, 0...."
2,33,1805,1,Farbkodierte Dopplersonographie und Power-Dopp...,Sono,"[-0.0011921226, 0.019474441, 0.0019958976, 0.0..."
3,33,1805,2,Farbkodierte Dopplersonographie und Power-Dopp...,Sono,"[-0.00827356, 0.020826263, -0.0036322814, 0.02..."
4,33,1805,3,Farbkodierte Dopplersonographie und Power-Dopp...,Sono,"[-0.006143069, 0.022493884, -0.005542918, 0.02..."
...,...,...,...,...,...,...
367135,37446,14926,194,AZ stabil. Durchfall etwas besser aber noch da...,clinical_assessment,"[-0.019216418, -0.006748805, 0.02392889, 0.028..."
367136,37446,14926,258,AZ gut. Medikation seit letztem Termin unverän...,clinical_assessment,"[-0.0041850563, 0.0056091207, 0.02928342, 0.02..."
367137,37446,14926,323,"CellCept-Reduktion am 20.04. von 2g auf 1,5g/T...",clinical_assessment,"[-0.01686095, 0.0029132883, -0.0016716092, 0.0..."
367138,37446,14926,397,Durchfall etwas besser. Corona: inzwischen 4x ...,clinical_assessment,"[-0.0026658918, -0.011106887, 0.031188088, 0.0..."


In [8]:
all_patients = set(static_df['patient_id'])

patients_with_notes = set(notes['patient_id'])

patients_no_notes = all_patients - patients_with_notes
patients_no_notes

{1125,
 5350,
 5364,
 5442,
 7007,
 7135,
 7237,
 8894,
 35021,
 35694,
 37928,
 38051,
 38056,
 38081}

In [20]:
# Load the notes dataframe (assuming it's already loaded in this environment or replace with actual file path)
# notes = pd.read_csv("path_to_notes_file.csv")

# Example dataframe structure (replace with actual loading mechanism if necessary)
columns = ["patient_id", "transplant_id", "rel_days", "text", "type", "embeddings"]
# Check if any patient has 0 notes
notes_per_patient = notes.groupby("patient_id").size()
patients_with_no_notes = notes_per_patient[notes_per_patient == 0]

# Check if any note lacks a valid embedding vector
invalid_embeddings = notes[notes["embeddings"].apply(lambda x:  np.mean(x) == 0)]

invalid_embeddings


,patient_id,transplant_id,rel_days,text,type,embeddings


In [6]:
def batch_embed_texts(encoder, texts, batch_size=32):
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        batch_embeddings = encoder(batch)
        all_embeddings.append(batch_embeddings.cpu().numpy())
    
    return np.concatenate(all_embeddings)

# Direct, simplified embedding generation
all_embeddings = batch_embed_texts(encoder, notes['text'].tolist())
np.save("../data/embeddings/emb_med_gte_simcse_en.npy", all_embeddings)

100%|██████████| 11474/11474 [1:59:48<00:00,  1.60it/s] 


## From here on old code, not necessary to run.

In [ ]:
%%script false --no-raise-error
loaded_embeddings = np.load("../data/embeddings/emb_gte.npy", allow_pickle=True)
notes['embeddings'] = list(loaded_embeddings) 
notes

In [ ]:
%%script false --no-raise-error
BATCH_SIZE = 32

class NotesDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

# Collate function to tokenize and chunk texts
def collate_fn(batch_texts):
    tokenized_texts = []
    text_chunk_mapping = []  # To track which text each chunk belongs to
    for i, text in enumerate(batch_texts):
        tokens = encoder.tokenizer.tokenize(text)
        #chunks = [" ".join(tokens[j:j+512]) for j in range(0, len(tokens), 512)]
        # or just take one chunk
        chunks = tokens[:512]
        tokenized_texts.extend(chunks)
        text_chunk_mapping.extend([i] * len(chunks))
    return tokenized_texts, text_chunk_mapping

# Create DataLoader
dataset = NotesDataset(notes['text'])
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)



In [ ]:
%%script false --no-raise-error
# Process texts in batches and compute embeddings
all_embeddings = [None] * len(notes)  # Placeholder for final embeddings
with torch.no_grad():
    for batch_chunks, text_chunk_mapping in tqdm(dataloader, desc="Computing embeddings in batches"):
        # Get embeddings for all chunks in the batch
        chunk_embeddings = encoder(batch_chunks).detach().cpu()

        # Aggregate embeddings back to their corresponding texts
        for i, text_index in enumerate(text_chunk_mapping):
            if all_embeddings[text_index] is None:
                all_embeddings[text_index] = []
            all_embeddings[text_index].append(chunk_embeddings[i])

# Compute the average embedding for each text
all_embeddings = [
    torch.stack(embeddings).mean(dim=0).numpy() if embeddings else np.zeros(encoder.model.config.hidden_size)
    for embeddings in all_embeddings
]

# Save the embeddings
all_embeddings = np.array(all_embeddings)
np.save("../data/embeddings/emb_gte.npy", all_embeddings)

In [ ]:
%%script false --no-raise-error
loaded_embeddings = np.load("../data/embeddings/emb_gte.npy", allow_pickle=True)
notes['embeddings'] = list(loaded_embeddings) 